# Thực nghiệm LLM-CRS trên Colab — ReDial + Qwen2.5-7B-Instruct

**Đọc trước khi bấm Run:**
- Đây là code THỰC NGHIỆM, backbone Qwen2.5-7B-Instruct là **phương án đề xuất, chưa phải quyết định chốt của GVHD**. Đổi backbone = đổi 1 dòng trong `config.yaml`.
- **Tập test chỉ chạy 1 lần cho cấu hình cuối.** Mọi thử nghiệm chỉnh prompt/siêu tham số chỉ làm trên tập **valid**.
- Kết quả chưa có thì bảng mục 6 trong bài báo vẫn để trống. Không bịa số, không chép số bài khác.
- Trước khi chạy: tải thư mục `code/` kèm theo lên Colab (kéo-thả vào Files ở cột trái), hoặc copy vào Google Drive rồi mount ở ô dưới.

## 0. Kiểm tra GPU + mount Drive (nếu dùng Drive)

In [ ]:
import torch, sys
print("Python:", sys.version)
print("GPU co san:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Ten GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print("CHUA CO GPU. Vao Runtime > Change runtime type > chon GPU (T4 tro len) roi chay lai.")


In [ ]:
# Chi chay neu Phong de code trong Google Drive, vi du /MyDrive/llm-crs/code
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/llm-crs/code
# print("Thu muc hien tai:")
# %pwd


## 1. Cài thư viện

In [ ]:
%pip install -q -r requirements.txt
print("Cai xong.")

## 2. Tải ReDial + đếm thật + chia tập (không cần GPU)
Chạy xong mở file `data/data_card.json` ra xem: số hội thoại / lượt nói / phim là số **đếm từ file đã tải**, đó mới là số ghi vào bài báo.

In [ ]:
!python 01_download_count.py

## 3. Tạo ví dụ đánh giá / huấn luyện (không cần GPU)

In [ ]:
!python 02_build_examples.py

## 4. Xem prompt trước — không tốn GPU
Đọc 2 prompt mẫu, chắc chắn là mình hiểu model sẽ thấy gì rồi mới chạy thật.

In [ ]:
!python 03_prompting_baseline.py --dry_run --n 2

## 5. Prompting baseline: 20 ví dụ valid, chế độ ranking (cần GPU, vài phút)
Model nhận danh sách 100 ứng viên và xếp top 10. Kết quả lưu ở `outputs/pred_valid_ranking_n20.jsonl`.

In [ ]:
!python 03_prompting_baseline.py --split valid --n 20 --mode ranking

## 6. Đánh giá lại (không cần GPU)

In [ ]:
!python 05_evaluate.py --predictions outputs/pred_valid_ranking_n20.jsonl

## 7. Thử bản RAG (retriever TF-IDF trên tên phim)
100 ứng viên lúc này do retriever tìm ra, không phải ngẫu nhiên. Nếu retriever không tìm ra đáp án đúng thì Recall ở ví dụ đó = 0 — đó là lỗi của retriever, ghi nhận thẳng.

In [ ]:
!python 03_prompting_baseline.py --split valid --n 20 --mode rag

## 8. Thử bản sinh tự do + đo ảo giác
Model tự gợi ý 3 phim + giải thích. Verifier kiểm tra từng tên phim có trong catalogue không.

In [ ]:
!python 03_prompting_baseline.py --split valid --n 20 --mode free

## 9. LoRA — chạy thử 200 ví dụ trước
Đừng chạy hết train ngay. Chạy thử 200 ví dụ, xem loss có giảm và adapter có lưu được không, rồi mới chạy hết. **Nhớ copy thư mục adapter lên Drive ngay sau khi xong, Colab ngắt là mất.**

In [ ]:
# !python 04_lora_train.py --max_examples 200 --epochs 1 --output_dir outputs/lora_smoke
print("Bo dau # o dong tren de chay thu LoRA 200 vi du.")

## 10. Sau khi chạy xong thì làm gì?
1. Mở `data/data_card.json` và các file `outputs/*_metrics.json`, copy số vào bảng mục 6 của bài báo — không gõ tay số khác.
2. Gửi lại cho mình 3 thứ: `data_card.json`, các file `_metrics.json`, và 1 dòng Phong tự viết: "Em hiểu code đang làm gì ở bước ...".
3. Bước tiếp theo (chưa làm trong notebook này): chạy lại baseline KBRD / KGSF / UniCRS bằng CRSLab trong môi trường riêng, rồi mới chạy tập test 1 lần cho cấu hình cuối.

## 9b. LoRA — chạy thật trên toàn bộ train (1 epoch, ~27.516 ví dụ)

Chỉ chạy sau khi chạy thử 200 ví dụ ở ô 9 xong và loss có giảm. Chạy này mất **nhiều giờ** trên T4 — Colab có thể ngắt giữa chừng, nhưng code lưu checkpoint mỗi 200 bước (thư mục `checkpoint-*` trong `outputs/lora_qwen25_7b`), adapter ở checkpoint cuối vẫn dùng được. **Trong lúc train, thỉnh thoảng copy `outputs/lora_qwen25_7b` lên Drive để khỏi mất.**

In [ ]:
!python 04_lora_train.py --epochs 1 --output_dir outputs/lora_qwen25_7b

## 9c. Đánh giá adapter LoRA vừa train (valid n=200)

Chạy 03 với `--adapter` để nạp adapter lên mô hình gốc. Kết quả lưu riêng tên có `_lora` để không ghi đè file của mô hình gốc.

In [ ]:
!python 03_prompting_baseline.py --split valid --n 200 --mode ranking --adapter outputs/lora_qwen25_7b

In [ ]:
!python 03_prompting_baseline.py --split valid --n 200 --mode free --adapter outputs/lora_qwen25_7b

In [ ]:
!python 05_evaluate.py --predictions outputs/pred_valid_ranking_lora_n200.jsonl
!python 05_evaluate.py --predictions outputs/pred_valid_free_lora_n200.jsonl

## 9d. Lưu kết quả lên Drive (làm ngay trước khi Colab ngắt)

Copy lên Drive: thư mục `outputs/lora_qwen25_7b` (adapter), các file `outputs/pred_valid_*_lora_n200*.jsonl`, các file `outputs/*_metrics.json`, thư mục `runs/`.